In [14]:
import cv2
import numpy as np
import plotly.graph_objects as go
import glob
import os

class VisualOdometry:
    def __init__(self, K, dist):
        self.K = K
        self.dist = dist
        
        # Inicializadores do OpenCV
        self.sift = cv2.SIFT_create()
        self.bf = cv2.BFMatcher(cv2.NORM_L2)
        
        # Estado do Sistema (O nosso Mapa)
        self.poses = []           # Lista de Matrizes de Projeção 3x4 (K[R|t])
        self.posicoes_cam = []    # Coordenadas (X,Y,Z) reais da câmera para plotagem
        self.mapa_3d = []         # Nuvem de pontos global (N, 3)
        self.mapa_descritores = []# Descritores SIFT associados a cada ponto 3D
        
        # Memória do frame anterior para triangulação
        self.kp_prev = None
        self.des_prev = None
        self.pose_prev = None

    def carregar_e_corrigir_imagem(self, caminho):
        """Carrega a imagem e remove a distorção da lente imediatamente."""
        img = cv2.imread(caminho, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Erro ao carregar {caminho}")
        # Remover a distorção da imagem inteira simplifica a matemática depois
        img_undistorted = cv2.undistort(img, self.K, self.dist)
        return img_undistorted

    def processar_primeiro_frame(self, img_path):
        """Frame 0: Apenas extrai features e define a origem do mundo."""
        img = self.carregar_e_corrigir_imagem(img_path)
        self.kp_prev, self.des_prev = self.sift.detectAndCompute(img, None)
        
        # Câmera 0 é a origem: [I | 0]
        Rt0 = np.hstack((np.eye(3), np.zeros((3, 1))))
        self.pose_prev = np.dot(self.K, Rt0) # Matriz de projeção P0
        
        self.poses.append(self.pose_prev)
        self.posicoes_cam.append(np.array([0, 0, 0]))
        print("Frame 0 processado (Origem definida).")

    def processar_segundo_frame(self, img_path):
        """Frame 1: Inicializa o Mapa 3D (Geometria Epipolar)."""
        img = self.carregar_e_corrigir_imagem(img_path)
        kp_curr, des_curr = self.sift.detectAndCompute(img, None)
        
        # 1. Match com o Frame 0
        matches = self.bf.knnMatch(self.des_prev, des_curr, k=2)
        pts0, pts1, bons_matches = [], [], []
        
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                pts0.append(self.kp_prev[m.queryIdx].pt)
                pts1.append(kp_curr[m.trainIdx].pt)
                bons_matches.append(m)
                
        pts0 = np.float32(pts0)
        pts1 = np.float32(pts1)
        
        # 2. Matriz Essencial e Pose (sem distorção, pois a imagem já foi corrigida)
        E, mask = cv2.findEssentialMat(pts0, pts1, self.K, method=cv2.RANSAC, prob=0.999, threshold=1.0)
        pts0 = pts0[mask.ravel() == 1]
        pts1 = pts1[mask.ravel() == 1]
        _, R, t, _ = cv2.recoverPose(E, pts0, pts1, self.K)
        
        # 3. Matriz de Projeção P1
        Rt1 = np.hstack((R, t))
        pose_curr = np.dot(self.K, Rt1)
        
        # 4. Triangulação
        pontos_4d = cv2.triangulatePoints(self.pose_prev, pose_curr, pts0.T, pts1.T)
        pontos_3d = (pontos_4d[:3, :] / pontos_4d[3, :]).T
        
        # 5. Salvar no Mapa Global
        self.mapa_3d.extend(pontos_3d)
        
        # Guardar os descritores do Frame 1 que geraram pontos 3D
        descritores_inliers = [des_curr[m.trainIdx] for i, m in enumerate(bons_matches) if mask.ravel()[i] == 1]
        self.mapa_descritores.extend(descritores_inliers)
        
        # Atualizar estado
        self.poses.append(pose_curr)
        self.posicoes_cam.append(-np.dot(R.T, t).flatten())
        self.kp_prev, self.des_prev, self.pose_prev = kp_curr, des_curr, pose_curr
        print(f"Frame 1 processado. {len(pontos_3d)} pontos 3D inicializados.")

    def processar_frame(self, img_path, frame_idx):
        """Frame N: Tracking com PnP e adição de novos pontos."""
        img = self.carregar_e_corrigir_imagem(img_path)
        kp_curr, des_curr = self.sift.detectAndCompute(img, None)
        
        # 1. Match do Frame Atual contra o MAPA GLOBAL 3D
        matches = self.bf.knnMatch(np.array(self.mapa_descritores), des_curr, k=2)
        
        pts_3d_conhecidos = []
        pts_2d_atuais = []
        
        for m, n in matches:
            if m.distance < 0.75 * n.distance:
                # queryIdx é o índice no mapa_descritores / trainIdx é o índice no frame atual
                pts_3d_conhecidos.append(self.mapa_3d[m.queryIdx])
                pts_2d_atuais.append(kp_curr[m.trainIdx].pt)
                
        if len(pts_2d_atuais) < 10:
            print(f"Frame {frame_idx}: Tracking perdido! Poucos inliers.")
            return

        pts_3d_conhecidos = np.float32(pts_3d_conhecidos)
        pts_2d_atuais = np.float32(pts_2d_atuais)
        
        # 2. Tracking: Achar a pose da Câmera N (PnP)
        # Dist passamos zerada pois a imagem já foi undistorted
        sucesso, rvec, tvec, inliers = cv2.solvePnPRansac(
            pts_3d_conhecidos, pts_2d_atuais, self.K, np.zeros(4)
        )
        
        if not sucesso:
            return
            
        R, _ = cv2.Rodrigues(rvec)
        Rt_curr = np.hstack((R, tvec))
        pose_curr = np.dot(self.K, Rt_curr)
        
        self.poses.append(pose_curr)
        self.posicoes_cam.append(-np.dot(R.T, tvec).flatten())
        
        # (Opcional) Aqui entraria a triangulação de NOVOS pontos entre Frame N-1 e N
        # para o mapa continuar crescendo. Omitido para focar no rastreamento base.
        
        self.kp_prev, self.des_prev, self.pose_prev = kp_curr, des_curr, pose_curr
        print(f"Frame {frame_idx} processado via PnP. Pose rastreada.")

    def plotar_trajetoria_e_mapa(self):
        """Visualização interativa 3D usando Plotly."""
        mapa = np.array(self.mapa_3d)
        cams = np.array(self.posicoes_cam)
        
        fig = go.Figure()
        
        # Nuvem de pontos (Filtra outliers muito distantes para o plot ficar limpo)
        distancias = np.linalg.norm(mapa, axis=1)
        mapa_limpo = mapa[distancias < np.percentile(distancias, 95)] 
        
        fig.add_trace(go.Scatter3d(
            x=mapa_limpo[:, 0], y=mapa_limpo[:, 1], z=mapa_limpo[:, 2],
            mode='markers', marker=dict(size=2, color='gray', opacity=0.5),
            name='Mapa 3D'
        ))
        
        # Trajetória da Câmera (Linha e Pontos)
        fig.add_trace(go.Scatter3d(
            x=cams[:, 0], y=cams[:, 1], z=cams[:, 2],
            mode='lines+markers', marker=dict(size=5, color='red'),
            line=dict(color='blue', width=4),
            name='Trajetória da Câmera'
        ))
        
        fig.update_layout(
            title='Odometria Visual - Trajetória e Nuvem de Pontos',
            scene=dict(
                xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
                yaxis=dict(autorange="reversed"),
                zaxis=dict(autorange="reversed"),
                aspectmode='data'
            ),
            margin=dict(l=0, r=0, b=0, t=40)
        )
        fig.show()

# ==========================================
# EXECUÇÃO DO PIPELINE
# ==========================================
if __name__ == "__main__":
    # 1. Parâmetros da sua Câmera (Do seu arquivo YAML)
    K = np.array([[765.4723, 0.0, 331.6566],
                  [0.0, 766.6782, 246.3781],
                  [0.0, 0.0, 1.0]], dtype=np.float64)
    
    dist = np.array([0.0844, 0.2218, 0.0, 0.0], dtype=np.float64)
    
    # 2. Configurar a lista de imagens
    # Substitua '*.jpg' pelo formato e pasta das suas imagens
    # O comando sorted garante que img_001 venha antes de img_002
    imagens = sorted(glob.glob('/home/aki/Desktop/GitHub/Python-VO/Datasets/dataset_20260321_120618/stereo_test/*.png')) 
    
    if len(imagens) < 2:
        print("Erro: Forneça pelo menos 2 imagens na pasta.")
    else:
        # 3. Inicializar a classe
        vo = VisualOdometry(K, dist)
        
        # 4. Rodar o Loop de Frames
        for i, img_path in enumerate(imagens):
            if i == 0:
                vo.processar_primeiro_frame(img_path)
            elif i == 1:
                vo.processar_segundo_frame(img_path)
            else:
                vo.processar_frame(img_path, i)
        
        # 5. Visualizar o Resultado
        vo.plotar_trajetoria_e_mapa()

Frame 0 processado (Origem definida).
Frame 1 processado. 394 pontos 3D inicializados.
Frame 2 processado via PnP. Pose rastreada.
Frame 3 processado via PnP. Pose rastreada.
Frame 4 processado via PnP. Pose rastreada.
Frame 5 processado via PnP. Pose rastreada.
Frame 13 processado via PnP. Pose rastreada.
Frame 15 processado via PnP. Pose rastreada.


In [18]:
# 1. Parâmetros da sua Câmera (Do seu arquivo YAML)
K = np.array([[8.1690378992770002e+02, 5.0510166700000003e-01, 6.0850726281690004e+02],
              [0.0, 8.1156803828490001e+02, 2.6347599764440002e+02],
              [0.0, 0.0, 1.0]], dtype=np.float64)

dist = np.array([-5.6143027800000002e-02, 1.3952563200000001e-01,
                 -1.2155906999999999e-03, -9.7281389999999998e-04,
                 -8.0878168799999997e-02], dtype=np.float64)

# 2. Configurar a pasta e o número de imagens a usar
pasta_imagens = '/home/aki/Desktop/GitHub/Python-VO/Datasets/urban27/stereo_left'
num_imagens = 1000  # ajustar para quantas imagens da sequência você quer usar

# Ordena pelo nome do arquivo, que é timestamp, para manter a sequência correta
imagens = sorted(glob.glob(os.path.join(pasta_imagens, '*.png')))
if len(imagens) == 0:
    raise ValueError(f"Nenhuma imagem encontrada em {pasta_imagens}")

imagens = imagens[:num_imagens]
print(f"Usando {len(imagens)} imagens de {pasta_imagens}")

if len(imagens) < 2:
    print("Erro: Forneça pelo menos 2 imagens na pasta.")
else:
    # 3. Inicializar a classe
    vo = VisualOdometry(K, dist)
    
    # 4. Rodar o Loop de Frames
    for i, img_path in enumerate(imagens):
        if i == 0:
            vo.processar_primeiro_frame(img_path)
        elif i == 1:
            vo.processar_segundo_frame(img_path)
        else:
            vo.processar_frame(img_path, i)
    
    # 5. Visualizar o Resultado
    vo.plotar_trajetoria_e_mapa()


Usando 1000 imagens de /home/aki/Desktop/GitHub/Python-VO/Datasets/urban27/stereo_left
Frame 0 processado (Origem definida).
Frame 1 processado. 1136 pontos 3D inicializados.
Frame 2 processado via PnP. Pose rastreada.
Frame 3 processado via PnP. Pose rastreada.
Frame 4 processado via PnP. Pose rastreada.
Frame 5 processado via PnP. Pose rastreada.
Frame 6 processado via PnP. Pose rastreada.
Frame 7 processado via PnP. Pose rastreada.
Frame 8 processado via PnP. Pose rastreada.
Frame 9 processado via PnP. Pose rastreada.
Frame 10 processado via PnP. Pose rastreada.
Frame 11 processado via PnP. Pose rastreada.
Frame 12 processado via PnP. Pose rastreada.
Frame 13 processado via PnP. Pose rastreada.
Frame 14 processado via PnP. Pose rastreada.
Frame 15 processado via PnP. Pose rastreada.
Frame 16 processado via PnP. Pose rastreada.
Frame 17 processado via PnP. Pose rastreada.
Frame 18 processado via PnP. Pose rastreada.
Frame 19 processado via PnP. Pose rastreada.
Frame 20 processado via